# Multiclass Classification

The previous lectures introduced classification: why linear regression fails for categorical outcomes, the logistic function, and the full sklearn workflow with `LogisticRegression`, `ColumnTransformer`, and `Pipeline`. All of that used binary classification - two classes, one decision boundary.

Most real problems have more than two classes. This notebook extends everything we've learned to multiclass classification using the MNIST handwritten digit dataset. Along the way we'll examine why accuracy alone is insufficient, introduce precision and recall, and compare several classification approaches including KNN and SVM.

Added / corrected in latest version:

- [x] example images of knn and svc boundaries
- [x] explain recall in threshold image
- [x] move gap in comparison to after accuracy
- [x] add stratified cross k-fold cv example for 5/not-5 (show effect)
- [x] fix 0→6 error in conf matrix interpretation
- [x] added Q&A about "splitting hairs" - small changes to eval metrics

## Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score,
)
from sklearn.dummy import DummyClassifier

### Load MNIST

MNIST is a set of 70,000 small images of digits handwritten by high school students and employees of the US Census Bureau. Each image is labeled with the digit it represents. It is considered the "hello world" of classification and provides an opportunity to explore how ML handles image data.

We load the data with `fetch_openml`, which returns a `Bunch` object - a dictionary-like container with `data`, `target`, and `DESCR` attributes.

In [ ]:
mnist = fetch_openml('mnist_784', as_frame=False)
X, y = mnist.data, mnist.target
print(f"Features: {X.shape}")
print(f"Labels:   {y.shape}")
print(f"Classes:  {np.unique(y)}")

The `DESCR` attribute provides background on the dataset, including how the train/test split is defined.

In [ ]:
print(mnist.DESCR)

> MNIST was published by LeCun, et al in 1998. Most recently, LeCun was the chief AI scientist at Meta Platforms (Facebook).
> 
> As I finalize this notebook on Mar 10, 2026, LeCun [announced Advanced Machine Intelligence to commercialize artificial intelligence systems built around reasoning, planning, and "world models"](https://www.reuters.com/business/ex-meta-ai-chief-yann-lecuns-ami-raises-103-billion-alternative-ai-approach-2026-03-10/). In that announcement it was revealed that LeCun and partners had raised \\$1.03 billion on a \\$3.50 billion pre-money valuation. (i.e., they sold 29.4% interest in the startup for \\$1.03 billion)

Each image is 28 × 28 = 784 pixels, stored as a flat vector of grayscale intensity values in the [0, 255] range. This is high-dimensional data - far more features than the handful of columns we used in regression.

The data for a single observation is presented below.

In [ ]:
X[0]

Our goal is to build a model that sees data like that as a "5". Seems far-fetched, no?

Let's see the data as humans normally do...

In [ ]:
# utility function to display a digit image
def plot_digit(image_data):
    image = image_data.reshape(28, 28)
    plt.imshow(image, cmap="binary")
    plt.axis("off")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(28, 28), cmap="binary")
    ax.set_title(f"Label: {y[i]}")
    ax.axis("off")
plt.suptitle("First 10 MNIST Images")
plt.tight_layout()
plt.show()

### Class Distribution

A balanced dataset has roughly equal representation across classes. Checking this up front tells us whether accuracy will be a meaningful metric.

In [ ]:
unique_values, counts = np.unique(y, return_counts=True)

plt.figure(figsize=(8, 4))
plt.bar(unique_values, counts)
plt.xlabel("Digit")
plt.ylabel("Count")
plt.title("Distribution of Digits in MNIST")
plt.grid(axis="y", alpha=0.3)
plt.show()

The classes are reasonably balanced - each digit represents roughly 10% of the data.

### Train-Test Split

The MNIST dataset has a predefined split: the first 60,000 images for training, the last 10,000 for testing.

In [ ]:
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

print(f"Training: {X_train.shape[0]:,} samples")
print(f"Test:     {X_test.shape[0]:,} samples")

## Binary Classification: 5 vs Not-5

Before tackling all 10 digits, we start with a simpler question: is this digit a 5 or not? Binary classification on MNIST lets us explore accuracy, baselines, and evaluation in a familiar setting before scaling up.

First, we'll create binary True / False labels for "5" / "not-5" cases from the existing data.

In [ ]:
y_train_5 = (y_train == '5')
y_test_5 = (y_test == '5')
print("first 5 rows:")
for idx in range(5):
    print(f"original value: {y_train[idx]}  new label: {y_train_5[idx]}")

print(f"\nTraining 5s: {y_train_5.sum():,} / {len(y_train_5):,} "
      f"({y_train_5.mean():.1%})")

Only about 9% of images are 5s. This is an imbalanced binary problem - similar to the Default dataset in the previous notebook, where only 3.3% of borrowers defaulted.

The `max_iter` parameter controls how many iterations the solver runs to find the optimal coefficients. The default (100) is often too low for high-dimensional data like MNIST's 784 features. Setting `max_iter=1000` gives the solver enough room to converge; you'll see a `ConvergenceWarning` if it runs out of iterations.

In [ ]:
model_5 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_5.fit(X_train, y_train_5)

train_acc = model_5.score(X_train, y_train_5)
test_acc = model_5.score(X_test, y_test_5)

print(f"Training accuracy: {train_acc:.4f}")
print(f"Test accuracy:     {test_acc:.4f}")

Over 97% - impressive? Let's check a baseline using `DummyClassifier`, which will always predict the majority class.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train_5)

print(f"Dummy accuracy:    {dummy.score(X_test, y_test_5):.4f}")

A model that predicts "not 5" for every image gets over 90% accuracy. This is the same lesson as the Default dataset: when one class dominates, accuracy is misleading. We need better tools.

## Beyond Accuracy

The confusion matrix breaks down predictions into four categories. We saw this in the previous notebook; now we formalize the metrics that come from it.

In [ ]:
y_pred_5 = model_5.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test_5, y_pred_5))
print()
print(classification_report(y_test_5, y_pred_5, target_names=["not-5", "5"]))

A note on `target_names`: the names map to the sorted unique values of the target. Since `y_test_5` is boolean, `False` (0) comes first and `True` (1) comes second - so `"not-5"` maps to `False` and `"5"` maps to `True`. This ordering convention applies whenever you use `target_names` in sklearn.

![](./images/09b-cm-homl2e-fig3_2.png)

### Interpreting the Results

Three metrics beyond accuracy, all derived from the confusion matrix:

- *Precision*: of all images the model called "5," what fraction actually were? Answers: "when the model says 5, can I trust it?"
- *Recall*: of all actual 5s, what fraction did the model catch? Answers: "is the model missing real 5s?"
- *F1 score*: the harmonic mean of precision and recall - a single number that balances both

$$
\text{Precision} = \frac{TP}{TP + FP} \qquad
\text{Recall} = \frac{TP}{TP + FN} \qquad
F_1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

These metrics address what accuracy hides: a model can be very accurate overall while completely failing on the class we care about.

Look at the "5" row in the classification report:

- *Precision 0.90*: when the model predicts "5," it's correct 90% of the time. 10% of predicted 5s are actually other digits (false positives).
- *Recall 0.84*: the model catches 84% of actual 5s. It misses 16% of real 5s, calling them "not 5" (false negatives).
- *F1 0.87*: the harmonic mean balances precision and recall into a single score.

Compare with the "not-5" row: precision 0.98, recall 0.99. The model is excellent at identifying non-5s but noticeably weaker at catching actual 5s. This asymmetry is invisible in the overall 97.7% accuracy.

The confusion matrix shows the raw counts behind these percentages: 747 true positives (correctly identified 5s), 145 false negatives (missed 5s), and 82 false positives (other digits incorrectly called 5).

### Precision-Recall Tradeoff

Precision and recall are in tension. Raising the decision threshold makes the model more selective - fewer predicted positives, so precision goes up, but recall drops because more real positives are missed. Lowering the threshold catches more positives (higher recall) but includes more false alarms (lower precision).

The right balance depends on the problem. A spam filter should favor precision - users tolerate missing some spam but hate losing real email. A cancer screening should favor recall - missing a real case is far worse than a false alarm.

sklearn's `predict` uses a fixed threshold of 0.5 for the class label decision. But the underlying probabilities from `predict_proba` let us make our own choice by manually creating predictions based on a given threshold and using the scoring functions directly.

In [ ]:
y_prob_5 = model_5.predict_proba(X_test)[:, 1]

# compare metrics at different thresholds
for threshold in [0.3, 0.5, 0.7]:
    # generate True / False predictions based on the generated probabilities and chosen threshold
    y_pred_t = (y_prob_5 >= threshold).astype(int)
    # call scoring functions directly to on the predicitions
    p = precision_score(y_test_5, y_pred_t)
    r = recall_score(y_test_5, y_pred_t)
    f1 = f1_score(y_test_5, y_pred_t)
    print(f"Threshold {threshold}: precision={p:.3f}  recall={r:.3f}  F1={f1:.3f}")

As expected: lowering the threshold from 0.5 to 0.3 improves recall at the cost of precision. Raising it to 0.7 does the opposite. This relationship is illustrated below.

![](./images/09b-pr-homl2e-fig3_3.png)

In this figure, the Precision for each threshold, $TP / (TP + FP)$, is based on the number of "5s" to the right (True Positives) divided by that number plus the number of "not-5s" to the *right* (False Positives)

| Threshold | TP | FP | Precision (TP / (TP + FP)) |
|:---------:|:--:|:--:|:--------------------------:|
| Low       |  6 |  2 | 6/8 = 75%                  |
| Mid       |  4 |  1 | 4/5 = 80%                  |
| High      |  3 |  0 | 3/3 = 100%                 |

The Recall for each threshold, $TP / (TP + FN)$, uses the same approach, except that the denominator includes False Negatives - the number of "5s" to the *left* of the threshold.

| Threshold | TP | FN | Recall (TP / (TP + FN)) |
|:---------:|:--:|:--:|:-----------------------:|
| Low       |  6 |  0 | 6/6 = 100%              |
| Mid       |  4 |  2 | 4/6 = 67%               |
| High      |  3 |  3 | 3/6 = 50%               |

***

##### When would you favor precision over recall?

It depends on the cost of each type of error:

- *Favor precision* (minimize false positives): spam filter, product recommendations, automated trading - wrong positive actions are costly or annoying
- *Favor recall* (minimize false negatives): cancer screening, fraud detection, equipment failure prediction - missing a real case is dangerous or expensive

In practice, domain experts set the threshold based on the business cost of each error type, not on a mathematical default. Every classification problem has an implicit *cost matrix*, even if nobody writes it down. The default 0.5 threshold assumes that false positives and false negatives cost the same - which is rarely true. Making those costs explicit turns threshold tuning from art into arithmetic. We will develop this idea further in upcoming lectures on classification metrics and imbalanced data.

***

### Threshold Tuning with `FixedThresholdClassifier`

The previous notebook noted that sklearn's `predict` always uses 0.5. As of sklearn 1.5, there is a cleaner way to override this: `FixedThresholdClassifier` wraps any classifier and applies a custom threshold at prediction time.

In [ ]:
from sklearn.model_selection import FixedThresholdClassifier

# wrap our pipeline with a lower threshold to catch more 5s
model_5_low = FixedThresholdClassifier(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    threshold=0.3,
)
model_5_low.fit(X_train, y_train_5)
y_pred_low = model_5_low.predict(X_test)

print("With threshold=0.3:")
print(classification_report(y_test_5, y_pred_low, target_names=["not-5", "5"], digits=3))

We've used `digits=3` to override the default rounding and give more precision in this report. The precision shown for the "5" class matches the 0.827 from our manual calculation above. Both approaches apply the same 0.3 threshold and produce the same predictions, they're just different ways to compute and display the same metrics. The `classification_report` is more convenient because it shows precision, recall, and F1 for *both* classes in one call, while the manual approach above uses individual scoring functions on the positive class only.

`FixedThresholdClassifier` makes threshold tuning part of the sklearn API rather than manual post-processing. In a later lecture we'll see `TunedThresholdClassifierCV`, which searches for the optimal threshold automatically using cross-validation.

## Multiclass Strategies with Logistic Regression

Now we move to the full 10-digit problem. There are three approaches to multiclass classification with logistic regression, all covered in the slides:

- *Multinomial*: a single model that handles all K classes simultaneously via the softmax function. This is sklearn's default for `LogisticRegression`.
- *One-vs-Rest (OvR)*: train K binary classifiers, each distinguishing one class from all others. Predict the class whose classifier is most confident.
- *One-vs-One (OvO)*: train K(K-1)/2 binary classifiers, one for each pair of classes. Predict by majority vote.

These are three *strategies* for the same underlying model. The logistic regression math is the same in each case; the difference is how the multiclass problem is decomposed.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

The scores below are single-split estimates from the predefined train/test partition - a point sample, not a distribution. We'll use cross-validation later for a more robust comparison.

First, Multinomial (the default): a single model handles all 10 classes via softmax:

In [ ]:
multi_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

t0 = time.time()
multi_clf.fit(X_train, y_train)
fit_multi = time.time() - t0

t0 = time.time()
train_multi = multi_clf.score(X_train, y_train)
test_multi = multi_clf.score(X_test, y_test)
score_multi = time.time() - t0

print(f"Multinomial - Train: {train_multi:.4f}  Test: {test_multi:.4f}  "
      f"(fit {fit_multi:.1f}s, score {score_multi:.1f}s)")

Next, One-vs-Rest: 10 binary classifiers, each separating one digit from the rest:

In [ ]:
ovr_clf = make_pipeline(
    StandardScaler(),
    OneVsRestClassifier(LogisticRegression(max_iter=1000)),
)

t0 = time.time()
ovr_clf.fit(X_train, y_train)
fit_ovr = time.time() - t0

t0 = time.time()
train_ovr = ovr_clf.score(X_train, y_train)
test_ovr = ovr_clf.score(X_test, y_test)
score_ovr = time.time() - t0

print(f"OvR         - Train: {train_ovr:.4f}  Test: {test_ovr:.4f}  "
      f"(fit {fit_ovr:.1f}s, score {score_ovr:.1f}s)")

Finally, One-vs-One: 45 binary classifiers, one for each pair of digits:

In [ ]:
ovo_clf = make_pipeline(
    StandardScaler(),
    OneVsOneClassifier(LogisticRegression(max_iter=1000)),
)

t0 = time.time()
ovo_clf.fit(X_train, y_train)
fit_ovo = time.time() - t0

t0 = time.time()
train_ovo = ovo_clf.score(X_train, y_train)
test_ovo = ovo_clf.score(X_test, y_test)
score_ovo = time.time() - t0

print(f"OvO         - Train: {train_ovo:.4f}  Test: {test_ovo:.4f}  "
      f"(fit {fit_ovo:.1f}s, score {score_ovo:.1f}s)")

***

##### Why does OvO train faster despite fitting 45 models?

Two competing effects: OvO fits far more models (45 vs 10 for OvR or 1 for multinomial), but each model trains on only the observations from two classes - roughly 12,000 samples instead of 60,000. The smaller data wins because training cost for most algorithms grows faster than linearly with sample size. Cutting the data to 1/5 makes each fit more than 5x cheaper, which more than compensates for fitting 45 models instead of one.

***

## Other Classifiers: KNN and SVM

The previous section compared three strategies using the same model (logistic regression). Now we compare entirely different models on the same problem.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

### KNN

We used `KNeighborsRegressor` for regression in a previous lecture. `KNeighborsClassifier` is the classification counterpart - same algorithm, same API (aka code interface / syntax), predicts class labels by majority vote of the k nearest neighbors. KNN is distance-based, so scaling is required (already established in 06b).

In [ ]:
knn_clf = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(),  # Simply specify the Classifier variant
)

t0 = time.time()
knn_clf.fit(X_train, y_train)
fit_knn = time.time() - t0

t0 = time.time()
train_knn = knn_clf.score(X_train, y_train)
test_knn = knn_clf.score(X_test, y_test)
score_knn = time.time() - t0

print(f"KNN         - Train: {train_knn:.4f}  Test: {test_knn:.4f}  "
      f"(fit {fit_knn:.1f}s, score {score_knn:.1f}s)")

When used for classification, KNN creates non-linear decision boundaries where complexity (overfitting) decreases with the number of neighbors, $K$. This is illustrated in the figures below.

> Note: the Bayes Decision Boundary identified below is the theoretical ideal. For two classes, it's the set of points where P(Y=1|X) = P(Y=0|X) = 0.5. In real-world cases it is unknown since calculating it requires the true population distributions. All real classifiers are trying to approximate it from data.

| | |
|:---:|:---:|
| ![](./images/09b-knn-isl-fig2_13.png) | ![](./images/09b-knn-isl-fig2_16.png) |

### SVM

Support Vector Machines take a different approach to classification: instead of modeling probabilities (logistic regression) or counting neighbors (KNN), SVM finds the decision boundary that maximizes the *margin* - the distance between the boundary and the nearest points from each class. The intuition: a wider margin means more confidence in the boundary's placement. The concept is illustrated below.

![Credit: ISL, figure 9.3](./images/09b-svm-isl-fig9_3.png)

`SVC(kernel='linear')` fits a linear boundary, similar in spirit to logistic regression but optimized for margin width rather than likelihood. The `kernel` parameter controls the shape of the boundary - `'rbf'` (the default) allows curved, non-linear boundaries. We'll explore SVM in more depth in a later lecture.

In [ ]:
svm_clf = make_pipeline(
    StandardScaler(),
    SVC(kernel='linear'),
)

t0 = time.time()
svm_clf.fit(X_train, y_train)
fit_svm = time.time() - t0

t0 = time.time()
train_svm = svm_clf.score(X_train, y_train)
test_svm = svm_clf.score(X_test, y_test)
score_svm = time.time() - t0

print(f"SVM         - Train: {train_svm:.4f}  Test: {test_svm:.4f}  "
      f"(fit {fit_svm:.1f}s, score {score_svm:.1f}s)")

SVM training is noticeably slower than the other methods on this dataset. The algorithm must solve a quadratic optimization problem that involves comparing pairs of training samples, so its cost grows roughly as $O(n^2)$ to $O(n^3)$ with sample size. On 60,000 high-dimensional images, that adds up. This is a practical consideration when choosing models - SVM's accuracy may not justify the training cost at this scale.

The $O(\cdot)$ notation - "Big-O" - describes how an algorithm's cost scales with input size $n$. An $O(n)$ algorithm doubles in cost when you double the data; an $O(n^2)$ algorithm quadruples. Most ML algorithms fall in the $O(n)$ to $O(n^3)$ range for training.

In [ ]:
# Big-O complexity comparison
n = np.linspace(1, 50, 200)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n, np.ones_like(n), label="$O(1)$", linewidth=2)
ax.plot(n, np.log2(n), label=r"$O(\log n)$", linewidth=2)
ax.plot(n, n, label="$O(n)$", linewidth=2)
ax.plot(n, n * np.log2(n), label=r"$O(n \log n)$", linewidth=2)
ax.plot(n, n**2, label="$O(n^2)$", linewidth=2)
ax.plot(n, n**3 / 50, label="$O(n^3)$", linewidth=2, linestyle="--")

ax.set_xlabel("Input size ($n$)")
ax.set_ylabel("Operations (relative)")
ax.set_title("Big-O Complexity Growth Rates")
ax.set_ylim(0, 300)
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Comparison

In [ ]:
results = pd.DataFrame({
    "Method": ["Multinomial", "OvR", "OvO", "KNN", "SVM (linear)"],
    "Train Accuracy": [train_multi, train_ovr, train_ovo, train_knn, train_svm],
    "Test Accuracy": [test_multi, test_ovr, test_ovo, test_knn, test_svm],
    "Fit Time (s)": [fit_multi, fit_ovr, fit_ovo, fit_knn, fit_svm],
    "Score Time (s)": [score_multi, score_ovr, score_ovo, score_knn, score_svm],
})
results["Total Time (s)"] = results["Fit Time (s)"] + results["Score Time (s)"]
results.insert(3, "Gap", results["Train Accuracy"] - results["Test Accuracy"])
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# accuracy comparison
x = np.arange(len(results))
width = 0.35
ax1.bar(x - width/2, results["Train Accuracy"], width, label="Train")
ax1.bar(x + width/2, results["Test Accuracy"], width, label="Test")
ax1.set_xticks(x)
ax1.set_xticklabels(results["Method"], rotation=30, ha="right")
ax1.set_ylabel("Accuracy")
ax1.set_title("Train vs Test Accuracy")
ax1.set_ylim(0.85, 1.0)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# time breakdown (stacked fit + score)
ax2.bar(x, results["Fit Time (s)"], width=0.5, label="Fit")
ax2.bar(x, results["Score Time (s)"], width=0.5, bottom=results["Fit Time (s)"], label="Score")
ax2.set_xticks(x)
ax2.set_xticklabels(results["Method"], rotation=30, ha="right")
ax2.set_ylabel("Time (s)")
ax2.set_title("Fit vs Score Time")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

Key observations:

- KNN achieves the highest test accuracy, suggesting non-linear decision boundaries matter for this data
- SVM (linear) has the largest train-test gap (0.053) - it fits the training data very well but generalizes less effectively than KNN or even OvO. High training accuracy paired with a large gap is a classic overfitting signal.
- The three logistic regression strategies perform similarly on this balanced dataset - the choice matters more when classes are imbalanced or boundaries are complex
- OvR has the smallest gap, suggesting the most stable generalization among the logistic regression strategies

***

##### Why is KNN's score time so much higher than its fit time?

KNN is a *lazy learner*: fitting just stores the training data (near-instant), while predicting requires computing distances from each test point to all 60,000 training points. Every other method here is an *eager learner* - it does the heavy work during fit (finding coefficients or support vectors) and prediction is a cheap matrix multiply. This is a fundamental tradeoff: KNN trades fast training for slow prediction. In production systems where prediction latency matters, this can be a dealbreaker.

***

## Evaluating the Multiclass Model

Accuracy tells us the overall error rate. The confusion matrix tells us *where* the model fails - which digits get confused with which.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold

### Cross-Validation

A single train/test split gives one estimate. Cross-validation gives a distribution.

In regression we used `KFold`, which splits data into k equal parts without regard to the target values. For classification, `StratifiedKFold` is usually a better choice - it ensures each fold preserves the class distribution of the full dataset. With imbalanced data this prevents folds where the minority class is absent or underrepresented. This ensures each fold is *representative* of the original data. Let's compare both using the base classifier.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

kf_acc = cross_val_score(multi_clf, X_train, y_train, cv=kf, scoring="accuracy")
skf_acc = cross_val_score(multi_clf, X_train, y_train, cv=skf, scoring="accuracy")

kf_f1 = cross_val_score(multi_clf, X_train, y_train, cv=kf, scoring="f1_macro")
skf_f1 = cross_val_score(multi_clf, X_train, y_train, cv=skf, scoring="f1_macro")

print("Accuracy:")
print(f"  KFold:           {kf_acc.mean():.4f} (+/- {kf_acc.std():.4f})")
print(f"  StratifiedKFold: {skf_acc.mean():.4f} (+/- {skf_acc.std():.4f})")
print()
print("F1 (macro):")
print(f"  KFold:           {kf_f1.mean():.4f} (+/- {kf_f1.std():.4f})")
print(f"  StratifiedKFold: {skf_f1.mean():.4f} (+/- {skf_f1.std():.4f})")

We include F1 alongside accuracy here because accuracy can mask problems with individual classes, as we saw earlier with the 5/not-5 baseline. The F1 score combines precision and recall into a single number; `f1_macro` averages F1 across all 10 classes equally, so a class the model struggles with drags the score down even if accuracy stays high. We'll cover F1 in more depth in the next lecture on classification metrics.

On MNIST the difference between `KFold` and `StratifiedKFold` is small for both metrics because the classes are roughly balanced (each digit ~10%). To see the effect more clearly, let's try both on the imbalanced 5-vs-not-5 problem where only ~9% of samples are positive:

In [ ]:
kf_f1_5 = cross_val_score(model_5, X_train, y_train_5, cv=kf, scoring="f1")
skf_f1_5 = cross_val_score(model_5, X_train, y_train_5, cv=skf, scoring="f1")

print("5-vs-not-5 (F1 score):")
print(f"  KFold:           {kf_f1_5.mean():.4f} (+/- {kf_f1_5.std():.4f})  per-fold: {kf_f1_5.round(4)}")
print(f"  StratifiedKFold: {skf_f1_5.mean():.4f} (+/- {skf_f1_5.std():.4f})  per-fold: {skf_f1_5.round(4)}")

With imbalanced classes, `StratifiedKFold` produces more consistent fold-to-fold scores (lower standard deviation) because each fold has the same proportion of 5s. Plain `KFold` can create folds with more or fewer 5s by chance, adding noise to the estimates. The means may be similar, but the variance tells the story - and with more extreme imbalance (1% positive), `KFold` can produce folds with *zero* positive cases, making those folds' metrics meaningless.

Always use `StratifiedKFold` for classification. On balanced data it behaves identically to `KFold`; on imbalanced data it prevents broken folds. There is no downside. Use plain `KFold` for regression (where there are no class proportions to preserve).

***

##### Are we splitting hairs? These differences seem tiny.

On MNIST, 0.001 in accuracy is about 10 images out of 10,000. That sounds negligible - and for digit recognition, it probably is. But the same 0.001 difference in a medical screening model means 10 patients who get a different diagnosis. Whether a difference is "small" depends entirely on what each prediction costs.

The more important number here is the *standard deviation*, not the mean. `StratifiedKFold`'s std was roughly 3x lower on the 5-vs-not-5 problem (0.006 vs 0.016). That's not about getting a better score - it's about getting a more *reliable estimate* of your score. If your CV variance is high, you can't trust your model comparison.

This is a theme we'll develop over the next two lectures: accuracy and small differences in accuracy can hide large differences in *what matters*. Classification metrics (precision, recall, F1) reveal what accuracy masks. And when we assign real costs to errors - dollars lost per missed fraud, lives affected per missed diagnosis - small probability differences become large practical differences. The tools for making this concrete are coming next.

***

### Confusion Matrix

We use `ConfusionMatrixDisplay` with normalization to show each cell as a percentage of the true class. This makes it easy to compare across digits regardless of class size. This is an important step for understanding the failure modes of your model and developing an intuition about what might improve it. The findings can support or undermine validation / verification.

In [ ]:
y_pred_multi = multi_clf.predict(X_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# all predictions (normalized by true class)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_multi,
    normalize="true", values_format=".0%", ax=ax1,
)
ax1.set_title("All Predictions")

# errors only (zero weight on correct predictions)
sample_weight = (y_pred_multi != y_test)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_multi,
    sample_weight=sample_weight,
    normalize="true", values_format=".0%", ax=ax2,
)
ax2.set_title("Errors Only")

plt.tight_layout()
plt.show()

The left matrix shows that most digits are classified correctly (bright diagonal), with 5s being the hardest (87% correct) and 1s the easiest (98%).

The error-only matrix on the right amplifies the mistakes. Some of the dominant confusions:

- 4 → 9 (43%): the most common error. Both have a vertical stroke; a closed loop on the 4 can look like the 9's curve.
- 1 → 8 (38%): some 1s with serifs or wide strokes get mistaken for 8s.
- 0 → 5 (38%): the round structure of 0 can resemble the bottom of a 5.
- 5 → 3 (29%) and 5 → 8 (27%): 5s share curved features with both digits.

These patterns make visual sense - the model struggles where humans would too. Tools like these help target improvements: if 4↔9 confusion dominates, we might engineer features that capture the difference, or try a model with more flexible decision boundaries.

### Exploring Predictions

For the multinomial model, `decision_function` returns one score per class - the raw linear score for each class:

$$z_k = \mathbf{w}_k^T \mathbf{x} + b_k$$

Higher scores mean more confidence. `predict_proba` converts these to probabilities via the softmax function, which generalizes the sigmoid to K classes:

$$P(y = k) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

For binary classification ($K = 2$), softmax reduces to the familiar sigmoid $\sigma(z) = \frac{1}{1 + e^{-z}}$ from the previous notebook.

In [ ]:
some_digit = X_test[:1]  # keep as 2D array

# decision function scores
z_values = multi_clf.decision_function(some_digit)[0]

# probabilities
probs = multi_clf.predict_proba(some_digit)[0]

# display sorted by confidence
df_scores = pd.DataFrame({
    "Class": multi_clf.classes_,
    "Decision Score": z_values,
    "Probability": probs,
}).sort_values("Decision Score", ascending=False)

print(f"True label: {y_test[0]}")
print(f"Predicted:  {multi_clf.predict(some_digit)[0]}")
print()
print(df_scores.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

`decision_function` returns raw scores (log-odds scale, unbounded); `predict_proba` returns calibrated probabilities that sum to 1. Both rank the classes the same way, but probabilities are easier to interpret and threshold. We will revisit the distinction between these two methods in the next lecture on classification metrics.

## Beyond Multiclass: Multilabel and Multioutput

Before we finish, a brief note on terminology and problem types.

What we've done in this notebook is *multiclass* classification: each image gets exactly one label from multiple possible classes (digits 0-9). Two related but distinct problem types:

- *Multilabel*: each instance can have multiple labels simultaneously. A photo might be tagged "outdoor" AND "dog" AND "sunny."
- *Multioutput*: each instance has multiple outputs, each of which may be multiclass. Predicting both the digit and the handedness of the writer from an image would be multioutput.

sklearn handles all three with the same API patterns. Here is a quick multilabel example using `KNeighborsClassifier` (which we just used above for multiclass).

In [ ]:
# create two binary labels for each digit
y_train_large = (y_train.astype(int) >= 7)  # large digit: 7, 8, 9
y_train_odd = (y_train.astype(int) % 2 == 1)  # odd digit: 1, 3, 5, 7, 9

# stack into a multilabel target (n_samples x 2)
y_train_multilabel = np.column_stack([y_train_large, y_train_odd])

print(f"Multilabel target shape: {y_train_multilabel.shape}")
print(f"First 5 labels (large, odd): {y_train_multilabel[:5]}")
print(f"First 5 digits:              {y_train[:5]}")

In [ ]:
knn_multi = KNeighborsClassifier()
knn_multi.fit(X_train, y_train_multilabel)

# predict both labels for the first test images
sample_pred = knn_multi.predict(X_test[:5])

for i in range(5):
    digit = y_test[i]
    large, odd = sample_pred[i]
    print(f"Digit {digit}: large={bool(large)}, odd={bool(odd)}")

The key takeaway: multiclass (one label, many classes) is the common case and the focus of this notebook. Multilabel and multioutput exist for more complex problems and use the same sklearn patterns.

## Summary

Key ideas from this notebook:

- *Multiclass* = one label per instance, multiple classes; *multilabel* = multiple labels per instance
- Three multiclass strategies with logistic regression: multinomial (default, single model), OvR (K models), OvO (K(K-1)/2 models)
- KNN and SVM are alternative classifiers with different inductive biases - KNN uses distance, SVM maximizes margin
- Accuracy is misleading for imbalanced classes - precision and recall reveal different aspects of model performance
- The precision-recall tradeoff is controlled by the decision threshold; `FixedThresholdClassifier` makes this part of the sklearn API
- Every classification problem has an implicit cost matrix - making error costs explicit is how we choose the right threshold
- Use `StratifiedKFold` for classification CV to preserve class proportions in each fold
- The confusion matrix reveals *where* the model fails, not just *how often*
- `decision_function` returns raw scores; `predict_proba` returns calibrated probabilities
- Pipeline and StandardScaler apply to all classifiers, same rules as regression